# Chapter 01 Laboratory — What Does It Mean for a Machine to Learn?

> ▶️ [Open in Google Colab](https://colab.research.google.com/github/manish7725/deeplearning/blob/reorg/class8-to-phd-curriculum/notebooks/01-what-is-learning.ipynb)
> 📖 [Read Chapter 01](https://github.com/manish7725/deeplearning/blob/reorg/class8-to-phd-curriculum/blogs/01-what-is-learning.md)

**Laboratory idea:** we will teach a tiny model the rule `y = 2x + 1` from examples.

Learning loop: **predict → measure → calculate direction → update → repeat**.

## 🎯 Objectives

By the end you should be able to:

- explain the difference between prediction and learning;
- calculate squared error by hand;
- implement a trainable line using NumPy;
- explain what `w`, `b`, loss, gradient and learning rate mean;
- visualize loss falling during training;
- change one training variable and predict its effect before running the experiment.

## 1. Our tiny dataset

We secretly generated the examples using `y = 2x + 1`. The learner does not receive that rule. It only sees examples.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.array([1., 2., 3., 4.])
y = np.array([3., 5., 7., 9.])

print('x =', x)
print('y =', y)

## 2. Predict before learning

Start with `w = 1` and `b = 0`. For `x = 3`, calculate on paper:

`y_hat = w*x + b = 1*3 + 0 = 3`

The correct answer is 7, so the prediction is wrong.

In [ ]:
w, b = 1.0, 0.0
y_hat = w * x + b
error = y_hat - y
loss = np.mean(error ** 2)

print('predictions:', y_hat)
print('errors:', error)
print('MSE:', loss)

## 3. The loss function

For one example:

$$L=(y-\hat y)^2$$

For many examples:

$$MSE=\frac{1}{n}\sum_i(y_i-\hat y_i)^2$$

A smaller MSE means the predictions are closer to the targets for this objective.

## 4. Where do the gradients come from?

For our model `y_hat = wx + b`, define `e = y_hat - y`. Then:

$$L=\frac{1}{n}\sum_i e_i^2$$

Because `e = wx + b - y`, the chain rule gives:

$$\frac{\partial L}{\partial w}=\frac{1}{n}\sum_i 2e_i x_i$$

and

$$\frac{\partial L}{\partial b}=\frac{1}{n}\sum_i 2e_i.$$

Do not memorize these yet. In later chapters we will derive the chain rule carefully. Here we use them to make the learning mechanism visible.

In [ ]:
dw = np.mean(2 * error * x)
db = np.mean(2 * error)
print('gradient with respect to w:', dw)
print('gradient with respect to b:', db)

## 5. Train from scratch with NumPy

Nothing is hidden inside a `fit()` call. Every important operation is visible.

In [ ]:
w, b = 0.0, 0.0
learning_rate = 0.01
losses = []

for step in range(2000):
    prediction = w * x + b
    error = prediction - y
    loss = np.mean(error ** 2)
    losses.append(loss)

    dw = np.mean(2 * error * x)
    db = np.mean(2 * error)

    w -= learning_rate * dw
    b -= learning_rate * db

print(f'w = {w:.4f}')
print(f'b = {b:.4f}')
print(f'prediction at x=5 = {w*5+b:.4f}')
assert abs(w - 2) < 0.01
assert abs(b - 1) < 0.01

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.xlabel('training step')
plt.ylabel('MSE loss')
plt.title('Learning: loss falls as parameters improve')
plt.show()

## 6. See the learned model

The dots are the examples. The line is the model after training.

In [ ]:
x_plot = np.linspace(0, 5, 100)
y_plot = w * x_plot + b

plt.figure(figsize=(8, 4))
plt.scatter(x, y, label='training examples')
plt.plot(x_plot, y_plot, label='learned model')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.title('The model learned a line from examples')
plt.show()

## 7. Interactive experiment: change the learning rate

The learning rate controls the size of each parameter update. Run the experiment with several values.

Try: `0.001`, `0.01`, `0.1`, and `0.5`.

**Predict first:** Which value will learn slowly? Which might become unstable?

In [ ]:
def train(lr, steps=100):
    w, b = 0.0, 0.0
    history = []
    for _ in range(steps):
        pred = w * x + b
        error = pred - y
        history.append(np.mean(error ** 2))
        dw = np.mean(2 * error * x)
        db = np.mean(2 * error)
        w -= lr * dw
        b -= lr * db
    return w, b, history

for lr in [0.001, 0.01, 0.1]:
    learned_w, learned_b, history = train(lr)
    print(f'lr={lr}: w={learned_w:.3f}, b={learned_b:.3f}, final loss={history[-1]:.4f}')

## 8. Failure experiment 💥

Try a very large learning rate such as `1.0`. If the loss explodes, ask why. The update is:

$$\theta_{new}=\theta_{old}-\eta\nabla L$$

A large `η` means a large jump. Gradient descent is not magic: it is a numerical procedure whose behaviour depends on the objective and step size.

## 9. Optional PyTorch bridge

Now let a framework calculate the gradient. The mathematics has not changed; PyTorch is automating the bookkeeping.

In [ ]:
import torch

tx = torch.tensor(x, dtype=torch.float32)
ty = torch.tensor(y, dtype=torch.float32)
tw = torch.tensor(0.0, requires_grad=True)
tb = torch.tensor(0.0, requires_grad=True)

for _ in range(2000):
    pred = tw * tx + tb
    loss = torch.mean((pred - ty) ** 2)
    loss.backward()
    with torch.no_grad():
        tw -= 0.01 * tw.grad
        tb -= 0.01 * tb.grad
        tw.grad.zero_()
        tb.grad.zero_()

print(f'w = {tw.item():.4f}, b = {tb.item():.4f}')
assert abs(tw.item() - 2) < 0.01
assert abs(tb.item() - 1) < 0.01

## 10. Scientist's notebook

Record your observations:

| Experiment | Prediction | Observation | Explanation |
|---|---|---|---|
| Small learning rate | | | |
| Medium learning rate | | | |
| Large learning rate | | | |

The goal is not to get the code to run. The goal is to explain **why** it behaved the way it did.

## 🧩 Challenges

1. Change the hidden rule to `y = 3x + 2` and train again.
2. Add noise to `y`. Does the model still discover approximately the underlying line?
3. Start with `w = 10` and `b = -10`. Does the algorithm still converge?
4. Remove the bias `b`. What kinds of lines can the model represent now?
5. Explain why a model with more parameters is not automatically a better model.

### Mini-project
Create a dataset from your own rule, train a model, plot the loss, visualize the learned function, and write a five-sentence explanation of the training process.

## 🔬 Research bridge

Our experiment is deliberately tiny, but the scientific structure scales:

`hypothesis → baseline → controlled change → measurement → explanation`.

Later chapters will replace a line with neural networks, but we will keep asking the same questions: What is the objective? What are the parameters? How is the gradient obtained? What causes failure? How do we know the improvement is real?

## ➡️ Next chapter

We used `x` as if everyone already knew how to represent information with numbers. Next we make that representation explicit.

**Chapter 02 — Numbers Become Vectors**: numbers become arrows, vectors become objects we can move and transform, and the language of linear algebra begins.